<a href="https://colab.research.google.com/github/Vishesh-Goyal7/NARAD-QC/blob/main/NARAD_QC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NARAD-QC
### Network Architecture for Routing, Authentication and Delivery in Quantum Commpunications

## Phase 0 : 06072026-16:15:32 - 06072026-16:48:53

Goal : Rebuild point to point CrypTon key generation and AES baseline.

Objective : Prove the original point-to-point system is working as the baseline.

In [1]:
# @title Dependency Installation for Phase 0

!pip install qiskit qiskit-aer pycryptodome pandas matplotlib --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00


In [2]:
# @title Imports and Global Config

import time
import secrets
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

from Crypto.Cipher import AES
from Crypto.Random import get_random_bytes

RNG = np.random.default_rng()
BACKEND = AerSimulator()

QBER_CUTOFF = 0.11
NONCE_LEN = 12

In [3]:
# @title AES-GCM Engine

def aes_gcm_encrypt(data: bytes, key: bytes) -> bytes:
    nonce = get_random_bytes(NONCE_LEN)
    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    ciphertext, tag = cipher.encrypt_and_digest(data)
    return nonce + ciphertext + tag


def aes_gcm_decrypt(blob: bytes, key: bytes) -> bytes:
    nonce = blob[:NONCE_LEN]
    rest = blob[NONCE_LEN:]
    ciphertext = rest[:-16]
    tag = rest[-16:]

    cipher = AES.new(key, AES.MODE_GCM, nonce=nonce)
    return cipher.decrypt_and_verify(ciphertext, tag)

In [4]:
# @title BB84 Step 1 : QRNG

def random_bitstring(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()


def random_bases(n: int) -> list[int]:
    return RNG.integers(0, 2, size=n).tolist()

In [5]:
# @title BB84 Step 2 : Qubit Encoding

def encode_qubits(bits: list[int], bases: list[int]) -> list[QuantumCircuit]:
    circuits = []

    for bit, basis in zip(bits, bases):
        qc = QuantumCircuit(1, 1)

        if bit == 1:
            qc.x(0)

        if basis == 1:
            qc.h(0)

        circuits.append(qc)

    return circuits

In [6]:
# @title BB84 Step 3 : Measure the Qubits

def measure_qubits(
    qubits: list[QuantumCircuit],
    bob_bases: list[int],
    eve_present: bool = False
) -> list[int]:

    measured_bits = []

    for i, prepared_qubit in enumerate(qubits):
        circuit = prepared_qubit.copy()

        if eve_present:
            eve_basis = int(RNG.integers(0, 2))

            eve_circuit = circuit.copy()

            if eve_basis == 1:
                eve_circuit.h(0)

            eve_circuit.measure(0, 0)

            transpiled_eve = transpile(eve_circuit, BACKEND)
            eve_result = BACKEND.run(
                transpiled_eve,
                shots=1,
                memory=True
            ).result()

            eve_bit = int(eve_result.get_memory()[0])

            circuit = QuantumCircuit(1, 1)

            if eve_bit == 1:
                circuit.x(0)

            if eve_basis == 1:
                circuit.h(0)

        if bob_bases[i] == 1:
            circuit.h(0)

        circuit.measure(0, 0)

        transpiled_bob = transpile(circuit, BACKEND)
        bob_result = BACKEND.run(
            transpiled_bob,
            shots=1,
            memory=True
        ).result()

        measured_bits.append(int(bob_result.get_memory()[0]))

    return measured_bits

In [7]:
# @title QBER and Cell Sifting and Key Reconciliation

def sift_keys(
    alice_bits: list[int],
    alice_bases: list[int],
    bob_bits: list[int],
    bob_bases: list[int]
) -> tuple[list[int], list[int]]:

    alice_sifted = []
    bob_sifted = []

    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:
            alice_sifted.append(alice_bits[i])
            bob_sifted.append(bob_bits[i])

    return alice_sifted, bob_sifted


def calculate_qber(alice_key: list[int], bob_key: list[int]) -> float:
    if len(alice_key) == 0:
        return 1.0

    errors = sum(a != b for a, b in zip(alice_key, bob_key))
    return errors / len(alice_key)

In [8]:
# @title One BB84 Round

def bb84_round(
    n_qubits: int = 256,
    eve_present: bool = False
) -> dict:

    alice_bits = random_bitstring(n_qubits)
    alice_bases = random_bases(n_qubits)
    bob_bases = random_bases(n_qubits)

    qubits = encode_qubits(alice_bits, alice_bases)
    bob_bits = measure_qubits(qubits, bob_bases, eve_present=eve_present)

    alice_sifted, bob_sifted = sift_keys(
        alice_bits,
        alice_bases,
        bob_bits,
        bob_bases
    )

    qber = calculate_qber(alice_sifted, bob_sifted)

    accepted = qber <= QBER_CUTOFF

    return {
        "alice_key": alice_sifted,
        "bob_key": bob_sifted,
        "sifted_length": len(alice_sifted),
        "qber": qber,
        "accepted": accepted,
        "eve_present": eve_present
    }

In [9]:
# @title Accumulate Session Key

def bits_to_bytes(bits: list[int]) -> bytes:
    usable_len = len(bits) - (len(bits) % 8)
    bits = bits[:usable_len]

    return bytes(
        int("".join(map(str, bits[i:i+8])), 2)
        for i in range(0, usable_len, 8)
    )


def generate_bb84_session_key(
    target_bits: int = 128,
    n_qubits_per_round: int = 256,
    max_rounds: int = 30,
    eve_present: bool = False,
    verbose: bool = True
) -> tuple[bytes, dict]:

    collected_bits = []
    round_logs = []

    start_time = time.perf_counter()

    for round_id in range(1, max_rounds + 1):
        result = bb84_round(
            n_qubits=n_qubits_per_round,
            eve_present=eve_present
        )

        round_logs.append({
            "round": round_id,
            "sifted_length": result["sifted_length"],
            "qber": result["qber"],
            "accepted": result["accepted"]
        })

        if verbose:
            print(
                f"Round {round_id}: "
                f"sifted={result['sifted_length']}, "
                f"QBER={result['qber']:.2%}, "
                f"accepted={result['accepted']}"
            )

        if result["accepted"]:
            collected_bits.extend(result["alice_key"])

        if len(collected_bits) >= target_bits:
            break

    elapsed = time.perf_counter() - start_time

    if len(collected_bits) < target_bits:
        raise RuntimeError(
            "Failed to generate enough accepted BB84 key bits."
        )

    final_bits = collected_bits[:target_bits]
    key = bits_to_bytes(final_bits)

    metadata = {
        "target_bits": target_bits,
        "rounds_used": len(round_logs),
        "accepted_rounds": sum(r["accepted"] for r in round_logs),
        "rejected_rounds": sum(not r["accepted"] for r in round_logs),
        "total_sifted_bits": sum(r["sifted_length"] for r in round_logs),
        "keygen_time_s": elapsed,
        "round_logs": round_logs
    }

    return key, metadata

In [10]:
# @title Phase 0 End to End System

message = b"NARAD-QC Phase 0: CrypTon baseline reconstruction."

session_key, meta = generate_bb84_session_key(
    target_bits=128,
    n_qubits_per_round=256,
    max_rounds=30,
    eve_present=False,
    verbose=True
)

ciphertext = aes_gcm_encrypt(message, session_key)
decrypted = aes_gcm_decrypt(ciphertext, session_key)

print("\nSession Key:", session_key.hex())
print("Key Metadata:", meta)
print("Ciphertext:", ciphertext.hex())
print("Decrypted:", decrypted.decode())
print("Roundtrip OK:", decrypted == message)

Round 1: sifted=136, QBER=0.00%, accepted=True

Session Key: 3160f4da001a105cc00876c0b887dd7c
Key Metadata: {'target_bits': 128, 'rounds_used': 1, 'accepted_rounds': 1, 'rejected_rounds': 0, 'total_sifted_bits': 136, 'keygen_time_s': 44.549718053000106, 'round_logs': [{'round': 1, 'sifted_length': 136, 'qber': 0.0, 'accepted': True}]}
Ciphertext: 966fee90b4fd90469ff9693c70883ea4cb3bd0dcda1b6bf79e65742345af348cd874cea2240c0d89ed31935b7580d89162442bd4f92e45596a9f39fa683317e582bbf9ab290de1b8f87b6b1a822e
Decrypted: NARAD-QC Phase 0: CrypTon baseline reconstruction.
Roundtrip OK: True


In [11]:
# @title Eve Presence Validation

try:
    eve_key, eve_meta = generate_bb84_session_key(
        target_bits=128,
        n_qubits_per_round=256,
        max_rounds=5,
        eve_present=True,
        verbose=True
    )

    print("Unexpectedly generated key with Eve present.")
    print(eve_meta)

except RuntimeError as e:
    print("Eve disrupted key generation as expected.")
    print(str(e))

Round 1: sifted=128, QBER=26.56%, accepted=False
Round 2: sifted=131, QBER=25.19%, accepted=False
Round 3: sifted=124, QBER=28.23%, accepted=False
Round 4: sifted=122, QBER=27.05%, accepted=False
Round 5: sifted=117, QBER=19.66%, accepted=False
Eve disrupted key generation as expected.
Failed to generate enough accepted BB84 key bits.
